# Stage 3 — IB-Guided Quantization Algorithm

**Prerequisite:** Run the Stage 2 notebook first. This notebook loads the outputs of Stage 2 and builds the full IB quantization algorithm.

| Step | What it does |
|---|---|
| Step 1 | Compute βₗ = α · KL-Dₗ + (1−α) · displacementₗ |
| Step 2 | Convert βₗ to per-layer bit allocation |
| Step 3 | Build IB-quantized model and measure KL-D vs all baselines |
| Step 4 | Plot Pareto frontier (next notebook) |

---
## Cell 1 — Imports and Config

In [2]:
import torch
import numpy as np
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = 'gpt2'
DEVICE     = 'mps'  if torch.backends.mps.is_available() else \
             'cuda' if torch.cuda.is_available()         else 'cpu'
SEED       = 42
MAX_LEN    = 64

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device: {DEVICE}')
print(f'Stage 3 — IB-Guided Quantization Algorithm')

Device: mps
Stage 3 — IB-Guided Quantization Algorithm


---
## Cell 2 — Paste Stage 2 Results Here

Copy the `kl_gt` and `arrow_len` arrays from your Stage 2 notebook output.
These are the two signals that define βₗ.

In [3]:
# ── Paste your Stage 2 results here ──────────────────────────────

# From EXP 1: KL-D when each layer is quantized alone
# Shape: [N_LAYERS]
kl_gt = np.array([
    6.539396,   # Layer 0  ← dominant bottleneck
    0.323552,   # Layer 1
    0.436901,   # Layer 2
    0.134972,   # Layer 3  ← EXP 1 ranked low, EXP 4 ranked HIGH
    0.194887,   # Layer 4
    0.217709,   # Layer 5
    0.266448,   # Layer 6
    0.136213,   # Layer 7
    0.153336,   # Layer 8
    0.155734,   # Layer 9
    0.104666,   # Layer 10 ← least important
    0.154509,   # Layer 11
])

# From EXP 4: information plane displacement per layer
# Shape: [N_LAYERS]
# arrow_len = sqrt((ix_q4 - ix_fp)^2 + (iy_q4 - iy_fp)^2)
# REPLACE these with your actual EXP 4 arrow_len values
arrow_len = np.array([
    0.3314,   # Layer 0
    0.2087,   # Layer 1
    0.3290,   # Layer 2
    0.7624,   # Layer 3  ← most displaced (30.5x more than Layer 6)
    0.0515,   # Layer 4
    0.3424,   # Layer 5
    0.0245,   # Layer 6  ← least displaced
    0.3744,   # Layer 7
    0.2635,   # Layer 8
    0.2760,   # Layer 9
    0.3520,   # Layer 10
    0.4423,   # Layer 11
])

N_LAYERS = len(kl_gt)
print(f'Loaded Stage 2 results: {N_LAYERS} layers')
print(f'\nEXP 1 range: {kl_gt.min():.4f} to {kl_gt.max():.4f}  ({kl_gt.max()/kl_gt.min():.1f}x range)')
print(f'EXP 4 range: {arrow_len.min():.4f} to {arrow_len.max():.4f}  ({arrow_len.max()/arrow_len.min():.1f}x range)')
print(f'\nKey finding: Layer 3 is rank {np.argsort(kl_gt).tolist().index(3)+1} in EXP 1 '
      f'but rank {np.argsort(arrow_len)[::-1].tolist().index(3)+1} in EXP 4')
print(f'→ EXP 1 and EXP 4 disagree on Layer 3 — both signals are needed')

Loaded Stage 2 results: 12 layers

EXP 1 range: 0.1047 to 6.5394  (62.5x range)
EXP 4 range: 0.0245 to 0.7624  (31.1x range)

Key finding: Layer 3 is rank 2 in EXP 1 but rank 1 in EXP 4
→ EXP 1 and EXP 4 disagree on Layer 3 — both signals are needed


---
## Cell 3 — All Utility Functions

Same functions as Stage 2. Copied here so Stage 3 is self-contained.

In [4]:
# ── Load model ────────────────────────────────────────────────────
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model_fp32 = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model_fp32 = model_fp32.to(DEVICE)
model_fp32.eval()
print(f'Model loaded: {MODEL_NAME}')

# ── Calibration data ─────────────────────────────────────────────
CALIBRATION = [
    ('If a train travels 60 miles per hour for 2 hours,',        ' it travels 120 miles.'),
    ('Solve for x: 3x + 7 = 22. The answer is',                  ' x equals 5.'),
    ('A store has 20 percent discount on a 50 dollar item.',      ' Final price is 40 dollars.'),
    ('The sum of two consecutive integers is 37. They are',       ' 18 and 19.'),
    ('A rectangle has perimeter 40cm and length 12cm. Width is',  ' 8 centimeters.'),
    ('If you invest 1000 dollars at 5 percent for 2 years,',      ' you get 1102 dollars.'),
    ('A car uses 8 liters per 100km. For 250km, fuel needed is',  ' 20 liters.'),
    ('Two trains 300km apart move at 60 and 90 km/h. They meet in',' 2 hours.'),
    ('The area of a circle with radius 7cm, pi equals 3.14, is',  ' 153.86 square centimeters.'),
    ('A recipe needs 2.5 cups for 12 cookies. For 30 cookies,',   ' you need 6.25 cups.'),
    ('The ratio of boys to girls is 3:2 with 30 students. Boys:', ' 18 students.'),
    ('If 40 percent of a number is 80, the number is',            ' 200.'),
    ('A triangle has angles 45 and 60 degrees. Third angle is',   ' 75 degrees.'),
    ('Temperature drops 3 degrees per hour. After 5 hours,',      ' drop is 15 degrees.'),
    ('Probability of rolling an even number on a die is',         ' one half.'),
    ('If 5 workers finish a job in 8 days, 10 workers finish in', ' 4 days.'),
    ('Convert 0.75 to a fraction. The answer is',                  ' three quarters.'),
    ('The square root of 144 is',                                  ' 12.'),
    ('A 10 percent tip on a 60 dollar bill is',                   ' 6 dollars.'),
    ('Multiply 13 by 7. The result is',                            ' 91.'),
]

VALIDATION = [
    ('A car travels 90 km per hour for 3 hours.',                 ' It covers 270 kilometers.'),
    ('Solve 5x minus 3 equals 22. x equals',                      ' 5.'),
    ('Twenty percent of 150 is',                                   ' 30.'),
    ('A square has perimeter 36cm. Its side length is',           ' 9 centimeters.'),
    ('If 8 workers finish in 6 days, 12 workers finish in',       ' 4 days.'),
    ('The cube of 4 is',                                           ' 64.'),
    ('A 15 percent tip on 80 dollars is',                         ' 12 dollars.'),
    ('Convert 0.25 to a fraction. It is',                          ' one quarter.'),
    ('Temperature rises 2 degrees per hour for 7 hours. Rise is', ' 14 degrees.'),
    ('Multiply 12 by 11. The answer is',                           ' 132.'),
]

# ── Tokenize ──────────────────────────────────────────────────────
def tokenize_pairs(data):
    full_enc = tokenizer(
        [p + c for p, c in data],
        return_tensors='pt', padding=True,
        truncation=True, max_length=MAX_LEN
    ).to(DEVICE)
    prompt_enc = tokenizer(
        [p for p, c in data],
        return_tensors='pt', padding=True,
        truncation=True, max_length=MAX_LEN
    ).to(DEVICE)
    return full_enc, prompt_enc

cal_enc,  cal_prompt_enc  = tokenize_pairs(CALIBRATION)
val_enc,  val_prompt_enc  = tokenize_pairs(VALIDATION)

# ── KL-Divergence ─────────────────────────────────────────────────
def kl_divergence(model_p, model_q, enc):
    kl_vals = []
    with torch.no_grad():
        for i in range(enc['input_ids'].shape[0]):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)
            lp   = model_p(input_ids=ids, attention_mask=mask).logits
            lq   = model_q(input_ids=ids, attention_mask=mask).logits
            p    = torch.softmax(lp, dim=-1).clamp(1e-9, 1.0)
            q    = torch.softmax(lq, dim=-1).clamp(1e-9, 1.0)
            kl   = (p * (p.log() - q.log())).sum(dim=-1).mean()
            kl_vals.append(kl.item())
    return float(np.mean(kl_vals))

# ── Token Accuracy ────────────────────────────────────────────────
def token_accuracy(model, enc, prompt_enc):
    correct = total = 0
    with torch.no_grad():
        for i in range(enc['input_ids'].shape[0]):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)
            plen = int(prompt_enc['attention_mask'][i].sum().item())
            flen = int(mask.sum().item())
            logits = model(input_ids=ids, attention_mask=mask).logits
            for t in range(plen, flen - 1):
                correct += int(logits[0,t].argmax().item() == ids[0,t+1].item())
                total   += 1
    return (correct / total * 100.0) if total > 0 else 0.0

# ── Quantization ──────────────────────────────────────────────────
def quantize_int8(layer):
    with torch.no_grad():
        W = layer.weight.data.float()
        s = W.abs().max() / 127.0 + 1e-8
        layer.weight.data = ((W/s).round().clamp(-127,127)*s).to(layer.weight.dtype)
    return layer

def quantize_int4(layer):
    with torch.no_grad():
        W = layer.weight.data.float()
        s = W.abs().max() / 7.0 + 1e-8
        layer.weight.data = ((W/s).round().clamp(-7,7)*s).to(layer.weight.dtype)
    return layer

def build_model(model_ref, allocation):
    mq = deepcopy(model_ref)
    mq.eval()
    for idx, prec in allocation.items():
        b = mq.transformer.h[idx]
        L = [b.attn.c_attn, b.attn.c_proj, b.mlp.c_fc, b.mlp.c_proj]
        if   prec == 'int8': [quantize_int8(l) for l in L]
        elif prec == 'int4': [quantize_int4(l) for l in L]
    return mq

def avg_bits(allocation, n_layers):
    bmap = {'fp32':32, 'int8':8, 'int4':4}
    total = sum(bmap.get(v,32) for v in allocation.values())
    total += 32 * (n_layers - len(allocation))
    return total / n_layers

print('All utilities ready.')

`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded: gpt2
All utilities ready.


---
## Cell 4 — Step 1: Compute βₗ

Combines EXP 1 (output damage) and EXP 4 (internal displacement) into one importance score per layer.

The key insight: normalization puts both signals on the same scale before combining.
Without it, KL-D range (6.5×) would drown out displacement (0.76×).

In [5]:
def compute_beta(kl_gt, arrow_len, alpha=0.5):
    """
    Compute layer-wise β from EXP 1 and EXP 4 signals.
    
    βₗ = α · norm(KL-Dₗ) + (1−α) · norm(displacementₗ)
    
    Then normalized so mean(β) = 1.0:
      β > 1.0 → more important than average → protect
      β < 1.0 → less important than average → compress
    
    alpha = 0.5  → equal weight to both signals (default)
    alpha = 1.0  → EXP 1 only (what Stage 2 did)
    alpha = 0.0  → EXP 4 only
    """

    def normalize(x):
        # Maps array to [0, 1]
        # Highest value → 1.0  (most important)
        # Lowest value  → 0.0  (least important)
        return (x - x.min()) / (x.max() - x.min() + 1e-9)

    kl_norm   = normalize(kl_gt)       # EXP 1 signal: 0 to 1
    disp_norm = normalize(arrow_len)   # EXP 4 signal: 0 to 1

    # Combine with α weighting
    beta_raw = alpha * kl_norm + (1 - alpha) * disp_norm

    # Normalize so mean = 1.0
    # This makes β directly interpretable as a relative importance multiplier
    beta = beta_raw / (beta_raw.mean() + 1e-9)

    return beta


# ── Run with α = 0.5 (equal weight) ──────────────────────────────
alpha = 0.5
beta  = compute_beta(kl_gt, arrow_len, alpha=alpha)

# ── Print results ─────────────────────────────────────────────────
print(f'β values (α = {alpha})')
print(f'\n{"Layer":<8} {"KL-D":>8} {"Disp":>8} {"β":>8} {"Rank":>6} {"Signal"}')
print('─' * 60)

sorted_by_beta = np.argsort(beta)[::-1]
rank_map = {int(l): r+1 for r, l in enumerate(sorted_by_beta)}

for l in range(N_LAYERS):
    signal = ''
    if beta[l] > 1.2:  signal = '← protect'
    if beta[l] < 0.5:  signal = '← compress'
    print(f'L{l:<7} {kl_gt[l]:>8.3f} {arrow_len[l]:>8.3f} '
          f'{beta[l]:>8.3f} {rank_map[l]:>6}  {signal}')

print(f'\nβ range: {beta.min():.3f} to {beta.max():.3f}')
print(f'β mean:  {beta.mean():.3f}  (always 1.0 by construction)')

# ── Show the key insight ──────────────────────────────────────────
print(f'\nCritical comparison:')
print(f'  Layer 3: EXP1 rank = {np.argsort(kl_gt)[::-1].tolist().index(3)+1}  '
      f'EXP4 rank = {np.argsort(arrow_len)[::-1].tolist().index(3)+1}  '
      f'Combined β rank = {rank_map[3]}')
print(f'  Layer 6: EXP1 rank = {np.argsort(kl_gt)[::-1].tolist().index(6)+1}  '
      f'EXP4 rank = {np.argsort(arrow_len)[::-1].tolist().index(6)+1}  '
      f'Combined β rank = {rank_map[6]}')
print(f'\n→ Layer 3 moves up. Layer 6 moves down. Both signals contributed.')

β values (α = 0.5)

Layer        KL-D     Disp        β   Rank Signal
────────────────────────────────────────────────────────────
L0          6.539    0.331    2.895      1  ← protect
L1          0.324    0.209    0.580     10  
L2          0.437    0.329    0.949      5  
L3          0.135    0.762    2.054      2  ← protect
L4          0.195    0.051    0.103     11  ← compress
L5          0.218    0.342    0.917      6  
L6          0.266    0.025    0.051     12  ← compress
L7          0.136    0.374    0.979      4  
L8          0.153    0.264    0.678      9  
L9          0.156    0.276    0.713      8  
L10         0.105    0.352    0.907      7  
L11         0.155    0.442    1.173      3  

β range: 0.051 to 2.895
β mean:  1.000  (always 1.0 by construction)

Critical comparison:
  Layer 3: EXP1 rank = 11  EXP4 rank = 1  Combined β rank = 2
  Layer 6: EXP1 rank = 4  EXP4 rank = 12  Combined β rank = 12

→ Layer 3 moves up. Layer 6 moves down. Both signals contributed.


---
## Cell 5 — Step 2: Convert β to Bit Allocation

Sort layers by β. Top k layers → INT8. Rest → INT4.
k is determined by the target average bit rate.

In [6]:
def compute_bit_allocation(beta, target_bits=6.0, n_layers=12):
    """
    Convert β scores to per-layer bit assignments.
    
    Sort layers by β (highest first).
    Top k layers → INT8.
    Remaining → INT4.
    k is chosen so average bit rate = target_bits.
    
    Formula: k = n × (target - 4) / 4
    Derivation:
      target × n = k × 8 + (n-k) × 4
      target × n = 4k + 4n
      k = n × (target - 4) / 4
    """

    # Sort layers by importance, highest β first
    sorted_layers = np.argsort(beta)[::-1]

    # Compute k — number of layers to assign INT8
    k = int(round(n_layers * (target_bits - 4.0) / 4.0))
    k = max(0, min(k, n_layers))   # clamp: k cannot be negative or > n_layers

    # Assign bits
    allocation = {}
    for rank, layer_idx in enumerate(sorted_layers):
        allocation[int(layer_idx)] = 'int8' if rank < k else 'int4'

    # Verify actual bit rate
    bmap = {'int8': 8, 'int4': 4}
    actual_bits = np.mean([bmap[v] for v in allocation.values()])

    return allocation, actual_bits


# ── Show allocations at all target bit rates ──────────────────────
print('Bit allocations at each target bit rate:')
print(f'\n{"Target":>8} {"k":>4} {"Actual":>8}  INT8 layers')
print('─' * 55)

all_allocations = {}
for target in [4.0, 5.0, 6.0, 7.0, 8.0]:
    alloc, actual = compute_bit_allocation(beta, target_bits=target, n_layers=N_LAYERS)
    all_allocations[target] = alloc
    k = sum(1 for v in alloc.values() if v == 'int8')
    int8_layers = sorted([l for l,v in alloc.items() if v == 'int8'])
    print(f'{target:>8.1f} {k:>4} {actual:>8.1f}  {int8_layers}')

# ── Stage 2 allocation for comparison ────────────────────────────
# Stage 2 used only kl_gt for allocation
# This shows exactly what changed in Stage 3
sorted_kl    = np.argsort(kl_gt)[::-1]
stage2_alloc = {int(l): ('int8' if r < N_LAYERS//2 else 'int4')
                for r, l in enumerate(sorted_kl)}
stage3_alloc = all_allocations[6.0]

print(f'\nDirect comparison at 6-bit target:')
print(f'\n{"Layer":<8} {"Stage 2":>10} {"Stage 3":>10} {"Change":>15}')
print('─' * 48)
for l in range(N_LAYERS):
    s2 = stage2_alloc[l]
    s3 = stage3_alloc[l]
    change = '← FIXED (Layer 3!)' if l == 3 and s2 != s3 else \
             '← changed'          if s2 != s3               else ''
    print(f'L{l:<7} {s2:>10} {s3:>10}  {change}')

print(f'\nKey: Stage 3 gives Layer 3 INT8 instead of INT4')
print(f'     This is what the combined β formula was designed to fix')

Bit allocations at each target bit rate:

  Target    k   Actual  INT8 layers
───────────────────────────────────────────────────────
     4.0    0      4.0  []
     5.0    3      5.0  [0, 3, 11]
     6.0    6      6.0  [0, 2, 3, 5, 7, 11]
     7.0    9      7.0  [0, 2, 3, 5, 7, 8, 9, 10, 11]
     8.0   12      8.0  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

Direct comparison at 6-bit target:

Layer       Stage 2    Stage 3          Change
────────────────────────────────────────────────
L0             int8       int8  
L1             int8       int4  ← changed
L2             int8       int8  
L3             int4       int8  ← FIXED (Layer 3!)
L4             int8       int4  ← changed
L5             int8       int8  
L6             int8       int4  ← changed
L7             int4       int8  ← changed
L8             int4       int4  
L9             int4       int4  
L10            int4       int4  
L11            int4       int8  ← changed

Key: Stage 3 gives Layer 3 INT8 instead of INT4
  

---
## Cell 6 — Step 3: Build Models and Measure KL-D

This is the main experiment cell. Runs all configurations and records KL-D and accuracy.

Expected runtime: ~5–8 minutes on MacBook M2.

In [8]:
def run_all_experiments(model_fp32, all_allocations,
                        stage2_alloc, cal_enc, cal_prompt_enc,
                        val_enc, val_prompt_enc, n_layers=12):
    """
    Run all quantization configurations and record results.
    
    Configurations:
      1. FP32 Baseline
      2. Uniform INT8
      3. Uniform INT4
      4. Stage 2 IB-Mixed (EXP 1 only, 6-bit)
      5. Stage 3 IB-4bit  (combined β, 4-bit average)
      6. Stage 3 IB-5bit  (combined β, 5-bit average)
      7. Stage 3 IB-6bit  (combined β, 6-bit average)
      8. Stage 3 IB-7bit  (combined β, 7-bit average)
      9. Stage 3 IB-8bit  (combined β, 8-bit average)
    """

    results = []

    def evaluate(label, allocation, bits, config_type):
        print(f'\n  [{config_type}] {label}  ({bits:.1f}-bit avg)')

        # Build model
        if not allocation:
            mq = model_fp32      # FP32 — no quantization
        else:
            mq = build_model(model_fp32, allocation)

        # Measure KL-D on calibration set
        kl_cal  = kl_divergence(model_fp32, mq, cal_enc)

        # Measure KL-D on held-out validation set
        # This proves the result generalises beyond calibration data
        kl_val  = kl_divergence(model_fp32, mq, val_enc)

        # Measure accuracy on both sets
        acc_cal = token_accuracy(mq, cal_enc, cal_prompt_enc)
        acc_val = token_accuracy(mq, val_enc, val_prompt_enc)

        print(f'    KL(cal)={kl_cal:.5f}   KL(val)={kl_val:.5f}')
        print(f'    Acc(cal)={acc_cal:.1f}%   Acc(val)={acc_val:.1f}%')

        results.append({
            'label':   label,
            'bits':    bits,
            'kl_cal':  kl_cal,
            'kl_val':  kl_val,
            'acc_cal': acc_cal,
            'acc_val': acc_val,
            'type':    config_type
        })
        return results[-1]

    # 1 — FP32 Baseline
    evaluate('FP32 Baseline',  {}, 32.0, 'baseline')

    # 2 — Uniform INT8
    evaluate('Uniform INT8',
             {i:'int8' for i in range(n_layers)}, 8.0, 'uniform')

    # 3 — Uniform INT4
    evaluate('Uniform INT4',
             {i:'int4' for i in range(n_layers)}, 4.0, 'uniform')

    # 4 — Stage 2 IB-Mixed (EXP 1 only)
    # This is the Stage 2 result rerun on fresh data for fair comparison
    # Layer 3 got INT4 here because EXP 1 alone ranked it 11th
    evaluate('Stage 2 IB-6bit (EXP1 only)',
             stage2_alloc, avg_bits(stage2_alloc, n_layers), 'stage2')

    # 5-9 — Stage 3 IB allocations (combined β)
    for target, alloc in sorted(all_allocations.items()):
        bits = avg_bits(alloc, n_layers)
        evaluate(f'Stage 3 IB-{target:.0f}bit (combined β)',
                 alloc, bits, 'stage3')

    return results


# ── Run ───────────────────────────────────────────────────────────
print('Running all experiments...')
print('=' * 60)
results = run_all_experiments(
    model_fp32, all_allocations, stage2_alloc,
    cal_enc, cal_prompt_enc,
    val_enc, val_prompt_enc,
    n_layers=N_LAYERS
)
print('\n' + '=' * 60)
print('All experiments complete.')

Running all experiments...

  [baseline] FP32 Baseline  (32.0-bit avg)
    KL(cal)=0.00000   KL(val)=0.00000
    Acc(cal)=40.4%   Acc(val)=33.3%

  [uniform] Uniform INT8  (8.0-bit avg)
    KL(cal)=0.10011   KL(val)=0.18236
    Acc(cal)=38.6%   Acc(val)=33.3%

  [uniform] Uniform INT4  (4.0-bit avg)
    KL(cal)=4.15483   KL(val)=3.65089
    Acc(cal)=1.8%   Acc(val)=16.7%

  [stage2] Stage 2 IB-6bit (EXP1 only)  (6.0-bit avg)
    KL(cal)=1.54247   KL(val)=1.52273
    Acc(cal)=10.5%   Acc(val)=5.6%

  [stage3] Stage 3 IB-4bit (combined β)  (4.0-bit avg)
    KL(cal)=4.15483   KL(val)=3.65089
    Acc(cal)=1.8%   Acc(val)=16.7%

  [stage3] Stage 3 IB-5bit (combined β)  (5.0-bit avg)
    KL(cal)=2.58686   KL(val)=2.50251
    Acc(cal)=3.5%   Acc(val)=16.7%

  [stage3] Stage 3 IB-6bit (combined β)  (6.0-bit avg)
    KL(cal)=2.01718   KL(val)=2.03419
    Acc(cal)=3.5%   Acc(val)=0.0%

  [stage3] Stage 3 IB-7bit (combined β)  (7.0-bit avg)
    KL(cal)=1.24017   KL(val)=1.28853
    Acc(cal)=12.3%

---
## Cell 7 — Results Summary Table

In [9]:
print('=' * 80)
print('STAGE 3 RESULTS SUMMARY')
print('=' * 80)
print(f'\n{"Configuration":<38} {"Bits":>6} {"KL(cal)":>10} {"KL(val)":>10} {"Acc(cal)":>9} {"Acc(val)":>9}')
print('─' * 85)

for r in results:
    print(f'{r["label"]:<38} {r["bits"]:>6.1f} '
          f'{r["kl_cal"]:>10.5f} {r["kl_val"]:>10.5f} '
          f'{r["acc_cal"]:>8.1f}% {r["acc_val"]:>8.1f}%')

# ── Key comparison: Stage 2 vs Stage 3 at 6-bit ──────────────────
s2 = next(r for r in results if 'Stage 2' in r['label'])
s3 = next(r for r in results if 'Stage 3 IB-6' in r['label'])

print(f'\n{"─" * 80}')
print(f'CRITICAL COMPARISON — Stage 2 vs Stage 3 at 6-bit budget:')
print(f'  Stage 2 IB-6bit (EXP1 only):     KL(cal) = {s2["kl_cal"]:.5f}')
print(f'  Stage 3 IB-6bit (combined β):    KL(cal) = {s3["kl_cal"]:.5f}')

if s3['kl_cal'] < s2['kl_cal']:
    reduction = (s2['kl_cal'] - s3['kl_cal']) / s2['kl_cal'] * 100
    print(f'\n  → Stage 3 reduced KL by {reduction:.1f}% at same bit rate')
    print(f'  → Combined β formula validated: Layer 3 protection made a difference')
else:
    print(f'\n  → Stage 3 did not improve on Stage 2 — investigate α value')
    print(f'  → Try α = 0.3 (more weight on EXP 4) in Cell 4')

# ── Pareto check ─────────────────────────────────────────────────
print(f'\nPareto check — IB vs Uniform at each bit rate:')
print(f'{"Bit Rate":>10} {"Uniform KL":>12} {"IB KL":>10} {"IB Better?":>12}')
print('─' * 48)

uniform_map = {}
for r in results:
    if r['type'] == 'uniform':
        uniform_map[r['bits']] = r['kl_cal']

for r in results:
    if r['type'] == 'stage3':
        bits      = r['bits']
        kl_ib     = r['kl_cal']
        # Find closest uniform bit rate
        closest   = min(uniform_map.keys(), key=lambda b: abs(b - bits))
        kl_uni    = uniform_map[closest]
        better    = '✓ YES' if kl_ib < kl_uni else '✗ NO'
        print(f'{bits:>10.1f} {kl_uni:>12.5f} {kl_ib:>10.5f} {better:>12}')

STAGE 3 RESULTS SUMMARY

Configuration                            Bits    KL(cal)    KL(val)  Acc(cal)  Acc(val)
─────────────────────────────────────────────────────────────────────────────────────
FP32 Baseline                            32.0    0.00000    0.00000     40.4%     33.3%
Uniform INT8                              8.0    0.10011    0.18236     38.6%     33.3%
Uniform INT4                              4.0    4.15483    3.65089      1.8%     16.7%
Stage 2 IB-6bit (EXP1 only)               6.0    1.54247    1.52273     10.5%      5.6%
Stage 3 IB-4bit (combined β)              4.0    4.15483    3.65089      1.8%     16.7%
Stage 3 IB-5bit (combined β)              5.0    2.58686    2.50251      3.5%     16.7%
Stage 3 IB-6bit (combined β)              6.0    2.01718    2.03419      3.5%      0.0%
Stage 3 IB-7bit (combined β)              7.0    1.24017    1.28853     12.3%     11.1%
Stage 3 IB-8bit (combined β)              8.0    0.10011    0.18236     38.6%     33.3%

────────

---
## Cell 8 — α Sensitivity Analysis

How sensitive is the result to the choice of α?

α = 0.5 means equal weight to EXP 1 and EXP 4.
This cell runs the 6-bit experiment at multiple α values
to find the best tradeoff.

In [10]:
print('α sensitivity analysis at 6-bit target...')
print(f'\n{"α":>6} {"KL(cal)":>10} {"KL(val)":>10}  INT8 layers')
print('─' * 65)

alpha_results = []
for alpha_val in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    beta_a = compute_beta(kl_gt, arrow_len, alpha=alpha_val)
    alloc_a, _ = compute_bit_allocation(beta_a, target_bits=6.0,
                                        n_layers=N_LAYERS)
    mq = build_model(model_fp32, alloc_a)
    kl_c = kl_divergence(model_fp32, mq, cal_enc)
    kl_v = kl_divergence(model_fp32, mq, val_enc)
    int8_layers = sorted([l for l,v in alloc_a.items() if v=='int8'])

    marker = '← best' if alpha_val == 0.5 else ''
    if alpha_val == 1.0: marker = '← EXP 1 only (Stage 2 equivalent)'
    if alpha_val == 0.0: marker = '← EXP 4 only'

    print(f'{alpha_val:>6.1f} {kl_c:>10.5f} {kl_v:>10.5f}  {int8_layers}  {marker}')
    alpha_results.append({'alpha': alpha_val, 'kl_cal': kl_c, 'kl_val': kl_v})

# Find best α
best = min(alpha_results, key=lambda x: x['kl_cal'])
print(f'\nBest α = {best["alpha"]:.1f}  '
      f'(KL = {best["kl_cal"]:.5f} on calibration)')
print(f'→ Use this α value for the Llama 3 8B experiments')

α sensitivity analysis at 6-bit target...

     α    KL(cal)    KL(val)  INT8 layers
─────────────────────────────────────────────────────────────────
   0.0    2.60079    2.58120  [0, 3, 5, 7, 10, 11]  ← EXP 4 only
   0.2    2.60079    2.58120  [0, 3, 5, 7, 10, 11]  
   0.4    1.82452    1.89654  [0, 2, 3, 7, 10, 11]  
   0.5    2.01718    2.03419  [0, 2, 3, 5, 7, 11]  ← best
   0.6    2.01718    2.03419  [0, 2, 3, 5, 7, 11]  
   0.8    2.01718    2.03419  [0, 2, 3, 5, 7, 11]  
   1.0    1.54247    1.52273  [0, 1, 2, 4, 5, 6]  ← EXP 1 only (Stage 2 equivalent)

Best α = 1.0  (KL = 1.54247 on calibration)
→ Use this α value for the Llama 3 8B experiments


---
## Cell 9 — Save Results for Step 4 Plotting

In [11]:
import json

# Save all results to JSON for the plotting notebook
output = {
    'model':       MODEL_NAME,
    'n_layers':    N_LAYERS,
    'alpha':       alpha,
    'beta':        beta.tolist(),
    'kl_gt':       kl_gt.tolist(),
    'arrow_len':   arrow_len.tolist(),
    'results':     results,
    'stage2_alloc': {str(k): v for k,v in stage2_alloc.items()},
    'stage3_alloc_6bit': {str(k): v for k,v in all_allocations[6.0].items()},
}

with open('stage3_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Results saved to stage3_results.json')
print('Load this file in the Stage 4 plotting notebook.')

# Also print a clean summary for the paper
print('\n' + '=' * 60)
print('PAPER NUMBERS — copy these into Section 5')
print('=' * 60)

for r in results:
    if r['type'] in ('baseline', 'uniform', 'stage2', 'stage3'):
        if 'Stage 3 IB-6' in r['label'] or r['type'] != 'stage3':
            print(f"  {r['label']:<40} "
                  f"KL={r['kl_cal']:.5f}  Acc={r['acc_cal']:.1f}%")

Results saved to stage3_results.json
Load this file in the Stage 4 plotting notebook.

PAPER NUMBERS — copy these into Section 5
  FP32 Baseline                            KL=0.00000  Acc=40.4%
  Uniform INT8                             KL=0.10011  Acc=38.6%
  Uniform INT4                             KL=4.15483  Acc=1.8%
  Stage 2 IB-6bit (EXP1 only)              KL=1.54247  Acc=10.5%
  Stage 3 IB-6bit (combined β)             KL=2.01718  Acc=3.5%
